In [2]:
import random
import json
import string
import re

# Define base examples and entity types
base_templates = [
    "The {EquipmentName} reported {FaultType} with code {FaultCode}.",
    "Observed {CustomerReportedSymptom} during {TimeDuration} driving in the {VehicleMakeModel}.",
    "Technician applied {MaintenanceMethod} to resolve the {DeviceProperties} issue.",
    "Readings showed {MeasurementValue} in the {VehicleComponentLocation}.",
    "Owner of the {VehicleMakeModel} reported {CustomerReportedSymptom} after a {FaultType} event.",
]

# Entity dictionary
entity_pool = {
    "EquipmentName": ["oxygen sensor", "brake pad", "throttle body", "gearbox", "fuel injector"],
    "FaultType": ["mechanical failure", "pressure drop", "electrical fault", "overheating"],
    "FaultCode": ["C0050", "P0301", "U0100", "B0028", "P0455"],
    "CustomerReportedSymptom": ["engine stalling", "brake squeal", "check engine light", "strange vibration"],
    "MaintenanceMethod": ["tightened connector", "replaced fuse", "reset ECU", "recalibrated sensor"],
    "DeviceProperties": ["voltage drop", "irregular signal", "high resistance", "sensor lag"],
    "MeasurementValue": ["12.5V", "78 psi", "150°C", "300 RPM"],
    "TimeDuration": ["10 minutes", "2 hours", "half an hour", "overnight"],
    "VehicleComponentLocation": ["left rear wheel", "engine bay", "dashboard", "rear axle"],
    "VehicleMakeModel": ["Toyota Camry", "Ford F-150", "Honda Civic", "BMW X5"]
}

# Typos / adversarial text injection
def introduce_typos(text, typo_prob=0.2):
    def typo(word):
        if len(word) < 3: return word
        idx = random.randint(0, len(word) - 2)
        return word[:idx] + word[idx+1] + word[idx] + word[idx+2:]

    return ' '.join([
        typo(word) if random.random() < typo_prob else word
        for word in text.split()
    ])

def replace_synonyms(text):
    synonyms = {
        "reported": ["noted", "mentioned", "observed"],
        "applied": ["performed", "conducted", "executed"],
        "issue": ["problem", "anomaly", "error"],
        "event": ["incident", "situation", "failure"],
        "readings": ["measurements", "logs", "data"],
    }
    for word, syns in synonyms.items():
        if word in text:
            text = text.replace(word, random.choice(syns))
    return text

# Entity annotation helper
def annotate_entities(text, entity_dict):
    entities = []
    for label, value in entity_dict.items():
        pattern = re.escape(value)
        match = re.search(pattern, text)
        if match:
            entities.append({
                "start": match.start(),
                "end": match.end(),
                "label": label
            })
    return entities

# Generate data
def generate_adversarial_dataset(n_samples=2000):
    data = []
    for _ in range(n_samples):
        template = random.choice(base_templates)

        # Sample values
        sampled = {label: random.choice(values) for label, values in entity_pool.items()}

        # Format and perturb sentence
        sentence = template.format(**sampled)
        sentence = replace_synonyms(sentence)
        sentence = introduce_typos(sentence)

        # Annotate
        entities = annotate_entities(sentence, sampled)

        data.append({
            "text": sentence,
            "entities": entities
        })
    return data

# Generate and save
adversarial_data = generate_adversarial_dataset(n_samples=2000)

# Save as JSON
with open("adversarial_synthetic_data.json", "w", encoding="utf-8") as f:
    json.dump(adversarial_data, f, indent=2, ensure_ascii=False)

print("✅ Generated 2000 adversarial examples and saved to adversarial_synthetic_data.json")


✅ Generated 2000 adversarial examples and saved to adversarial_synthetic_data.json
